In [1]:
import sys
import os
import sys
import os

# Add the project root to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Added {project_root} to sys.path")
sys.path.append(project_root + '/droid_slam')


Added /home/campus.ncl.ac.uk/c4071391/Projects/DROID-SLAM to sys.path


In [3]:
import collections

import torch
import torch.nn as nn
import onnx

from droid_slam.droid_net import DroidNet

# --- 1. Load the PyTorch Model ---
pth_model_path = '../droid.pth'

# Prefer exporting on CPU for maximum ONNX-compatibility.
# (You can set this to "cuda" if you specifically want GPU export.)
export_device = torch.device("cpu")

model = DroidNet()

ckpt = torch.load(pth_model_path, map_location="cpu", weights_only=True)
# DROID checkpoints can be stored as a plain state_dict or nested under a key.
if isinstance(ckpt, dict) and any(k in ckpt for k in ["state_dict", "model"]):
    state_dict = ckpt.get("model", ckpt.get("state_dict"))
else:
    state_dict = ckpt

# Strip DataParallel prefix if present
new_state_dict = collections.OrderedDict([
    (k.replace("module.", ""), v) for (k, v) in state_dict.items()
])

# Some checkpoints have 3 channels for delta/weight heads; this repo expects 2.
for key in [
    "update.weight.2.weight",
    "update.weight.2.bias",
    "update.delta.2.weight",
    "update.delta.2.bias",
]:
    if key in new_state_dict:
        new_state_dict[key] = new_state_dict[key][:2]

missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval().to(export_device)



Missing keys: []
Unexpected keys: []


/tmp/ipykernel_574854/3287948243.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(pth_model_path, map_location="cpu")


DroidNet(
  (fnet): BasicEncoder(
    (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (conv1): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (relu1): ReLU(inplace=True)
    (layer1): Sequential(
      (0): ResidualBlock(
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (relu): ReLU(inplace=True)
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      )
      (1): ResidualBlock(
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (relu): ReLU(inplace=True)
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affin

In [4]:
# --- 2. Create dummy inputs (used for ONNX tracing) ---
# NOTE: DroidNet expects images shaped [batch, num_frames, 3, H, W]
# In the original codebase these are usually RGB in [0,255] (float/uint8).

BATCH = 1
NUM_FRAMES = 3
C = 3
H = 240
W = 320

# Keep dummy inputs on the same device as the model for tracing.
device = next(model.parameters()).device

images = (torch.rand(BATCH, NUM_FRAMES, C, H, W, device=device) * 255.0).to(torch.float32)

# For the update module we operate at 1/8 resolution.
H8, W8 = H // 8, W // 8

# Dummy edge count (K) — matches graph_to_edge_list({i:[i]}) giving K==NUM_FRAMES
K = NUM_FRAMES

# We'll use model.cnet output to build realistic net/inp tensors.
with torch.no_grad():
    # DroidNet.extract_features includes normalization + tanh/relu split
    fmaps, net_all, inp_all = model.extract_features(images)

net = net_all[:, :K].contiguous()  # [B, K, 128, H/8, W/8]
inp = inp_all[:, :K].contiguous()  # [B, K, 128, H/8, W/8]

# CorrBlock output has 4 levels * (2r+1)^2 channels, with r=3 => 4*49 = 196
CORR_CH = 196
corr = torch.randn(BATCH, K, CORR_CH, H8, W8, device=device)

# Flow/motion input to UpdateModule has 4 channels
flow = torch.randn(BATCH, K, 4, H8, W8, device=device)


In [5]:

# --- 3. Export sub-models (fnet, cnet, update) to ONNX ---

# Output paths
fnet_onnx_path = '../fnet.onnx'
cnet_onnx_path = '../cnet.onnx'
update_onnx_path = '../update_core.onnx'

OPSET = 17

class _NormalizeImages(nn.Module):
    def __init__(self):
        super().__init__()
        mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 1, 3, 1, 1)
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, images):
        # images: [B, N, 3, H, W] in RGB, range [0,255]
        images = images[:, :, [2, 1, 0]] / 255.0  # to BGR + scale
        return (images - self.mean) / self.std


class FNetONNX(nn.Module):
    def __init__(self, fnet):
        super().__init__()
        self.norm = _NormalizeImages()
        self.fnet = fnet

    def forward(self, images):
        x = self.norm(images)
        return self.fnet(x)  # [B, N, 128, H/8, W/8]


class CNetSplitONNX(nn.Module):
    def __init__(self, cnet):
        super().__init__()
        self.norm = _NormalizeImages()
        self.cnet = cnet

    def forward(self, images):
        x = self.norm(images)
        out = self.cnet(x)  # [B, N, 256, H/8, W/8]
        net, inp = out.split([128, 128], dim=2)
        net = torch.tanh(net)
        inp = torch.relu(inp)
        return net, inp


class UpdateCoreONNX(nn.Module):
    def __init__(self, update):
        super().__init__()
        self.update = update

    def forward(self, net, inp, corr, flow):
        # Call UpdateModule with ii=None so we avoid graph aggregation
        net_out, delta, weight = self.update(net, inp, corr, flow)
        return net_out, delta, weight


# Wrap modules so ONNX inputs match DroidNet usage
fnet_export = FNetONNX(model.fnet).to(device).eval()
cnet_export = CNetSplitONNX(model.cnet).to(device).eval()
update_export = UpdateCoreONNX(model.update).to(device).eval()

print('Exporting fnet...')
torch.onnx.export(
    fnet_export,
    (images,),
    fnet_onnx_path,
    export_params=True,
    opset_version=OPSET,
    do_constant_folding=True,
    input_names=['images'],
    output_names=['fmaps'],
    dynamic_axes={
        'images': {0: 'batch', 1: 'frames', 3: 'height', 4: 'width'},
        'fmaps': {0: 'batch', 1: 'frames', 3: 'h8', 4: 'w8'},
    },
)
print(f'Saved: {fnet_onnx_path}')

print('Exporting cnet (split into net/inp)...')
torch.onnx.export(
    cnet_export,
    (images,),
    cnet_onnx_path,
    export_params=True,
    opset_version=OPSET,
    do_constant_folding=True,
    input_names=['images'],
    output_names=['net', 'inp'],
    dynamic_axes={
        'images': {0: 'batch', 1: 'frames', 3: 'height', 4: 'width'},
        'net': {0: 'batch', 1: 'frames', 3: 'h8', 4: 'w8'},
        'inp': {0: 'batch', 1: 'frames', 3: 'h8', 4: 'w8'},
    },
)
print(f'Saved: {cnet_onnx_path}')

print('Exporting update core (no CorrBlock, no graph aggregation)...')
# NOTE: `corr` is treated as an input to the update model because CorrBlock uses
# a custom CUDA extension (`droid_backends`) which cannot be exported to ONNX.
torch.onnx.export(
    update_export,
    (net, inp, corr, flow),
    update_onnx_path,
    export_params=True,
    opset_version=OPSET,
    do_constant_folding=True,
    input_names=['net', 'inp', 'corr', 'flow'],
    output_names=['net_out', 'delta', 'weight'],
    dynamic_axes={
        'net': {0: 'batch', 1: 'edges', 3: 'h8', 4: 'w8'},
        'inp': {0: 'batch', 1: 'edges', 3: 'h8', 4: 'w8'},
        'corr': {0: 'batch', 1: 'edges', 3: 'h8', 4: 'w8'},
        'flow': {0: 'batch', 1: 'edges', 3: 'h8', 4: 'w8'},
        'net_out': {0: 'batch', 1: 'edges', 3: 'h8', 4: 'w8'},
        'delta': {0: 'batch', 1: 'edges', 2: 'h8', 3: 'w8'},
        'weight': {0: 'batch', 1: 'edges', 2: 'h8', 3: 'w8'},
    },
)
print(f'Saved: {update_onnx_path}')

# --- 4. Verify ONNX files ---
for name, path in [('fnet', fnet_onnx_path), ('cnet', cnet_onnx_path), ('update_core', update_onnx_path)]:
    m = onnx.load(path)
    onnx.checker.check_model(m)
    print(f'ONNX model check passed: {name}')

print('Done.')



Exporting fnet...
Saved: ../fnet.onnx
Exporting cnet (split into net/inp)...


/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/droidenv/lib/python3.9/site-packages/torch/onnx/symbolic_helper.py:1454: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  warnings.warn(


Saved: ../cnet.onnx
Exporting update core (no CorrBlock, no graph aggregation)...
Saved: ../update_core.onnx
ONNX model check passed: fnet
ONNX model check passed: cnet
ONNX model check passed: update_core
Done.
